In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# How to Manage Long conversations

We have seen earlier in module 2 that we can create a short term memory for our agents through checkpoints, but when it's getting bigger, way more long conversations, then our context window of the agent filled up very quickly due to multiple threads. Either we would need to use larger context window model if we have the budget or we may need to implement below things which Langchain provide it inbuilt, 

1. Summarize Messages

2. Remove the unnecessary messages which we do not need, like ToolMessages.


By including these items into our agent, called a Middleware. Something like, before and after agent calls. 

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="claude-sonnet-4-5",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="claude-haiku-4-5",
            trigger=("tokens", 100), # it says the threshold or checkpoint tokens reaches 100 then trigger the summarizer middleware
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
         HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
    ]},
    config={"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is asking factual and speculative questions about a fictional lunar settlement called Lunapolis, including its status as the moon's capital, weather conditions, population of cheese miners, and potential labor disputes.\n\n## SUMMARY\n- Lunapolis has been established as the capital of the moon\n- Current weather in Lunapolis: clear skies, high of 120°C, low of -100°C\n- Population includes 100,000 cheese miners\n- Cheese miners' union is predicted to strike due to dissatisfaction with the new president\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone - the conversation appears to be a series of informational questions that have been answered. No further action or task completion is required unless the user provides additional questions or requests.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='dc92a370-887e-4954-96aa-48c54f20aa29'),
             

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is asking factual and speculative questions about a fictional lunar settlement called Lunapolis, including its status as the moon's capital, weather conditions, population of cheese miners, and potential labor disputes.

## SUMMARY
- Lunapolis has been established as the capital of the moon
- Current weather in Lunapolis: clear skies, high of 120°C, low of -100°C
- Population includes 100,000 cheese miners
- Cheese miners' union is predicted to strike due to dissatisfaction with the new president

## ARTIFACTS
None

## NEXT STEPS
None - the conversation appears to be a series of informational questions that have been answered. No further action or task completion is required unless the user provides additional questions or requests.


## Trim / Delete Messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messsages from the state"""
    messages = state["messages"]
    
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="claude-sonnet-4-5",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages]
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
    ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='1b0c888d-fbc8-4501-8de5-113c0fd35747'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='555a39dd-5926-455d-9652-66149e9e3911', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='e9e7e94d-0d1d-4056-a8b7-5bf9283a9e4d'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='44d917ff-b11b-4224-a4ce-ab842b1a8515', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='610fb51f-6918-439f-b68d-a4d8c1ecbf8b'),
              AIMessage(content="I need to clarify - are you asking me what the temperature is,

In [8]:
print(response["messages"][-1].content)

I need to clarify - are you asking me what the temperature is, or are you telling me information about the device's temperature?

If you're asking what you should check: **Feel the device to see if it's unusually hot or cold.** An overheated device might have shut down for safety, while an unusually cold device (if it's been in cold storage) might not power on properly.

Also, I still need to know: **Is the device showing any lights or indicators at all when you try to turn it on?** This will help me determine if it's getting power or if there's another issue.
